In [1]:
import os
import sys
import glob

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from pycaret.clustering import setup, create_model, assign_model, models, pull
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from datetime import datetime
import warnings
import time

from utils import preprocessing

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\tj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
warnings.filterwarnings('ignore')
DATA_PATH = '../data'
OUTPUT_DIR = '../results'
IMAGE_DIR = '../images'
    
document_df = preprocessing.get_default_data()

📂 51개 파일 발견
🔄 텍스트 전처리 시작...
   옵션: HTML제거=True, URL제거=True, 숫자제거=True
   옵션: 불용어제거=True, Lemmatization=True, Stemming=False
✅ 전처리 완료:
   - 원본 문서 수: 51
   - 제거된 빈 문서: 0
   - 최종 문서 수: 51
   - 평균 단어 수: 1266.9


In [3]:
# Cluster별 대표 문서와 유사 문서 Top-N 추출

from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics.pairwise import cosine_similarity

def auto_kmeans_with_representatives(document_df, text_column='processed_text',
                                     max_features=5000, svd_dim=50, k_range=(2,10), top_n=5):
    """
    document_df만 넣으면 자동으로:
    1. TF-IDF 벡터화
    2. SVD 차원 축소
    3. KMeans 여러 k 평가 (Silhouette, Davies-Bouldin, Calinski-Harabasz)
    4. 최적 k 선택
    5. 클러스터별 대표 문서 및 유사 문서 Top-N 추출
    
    Parameters:
        document_df (pd.DataFrame): 텍스트 데이터프레임
        text_column (str): 텍스트 컬럼명
        max_features (int): TF-IDF 최대 특성 수
        svd_dim (int): SVD 차원 축소 크기
        k_range (tuple): KMeans k 탐색 범위
        top_n (int): 클러스터별 유사 문서 Top-N 개수
    
    Returns:
        dict: 평가 지표, 최적 k, 클러스터별 대표 문서 및 유사 문서
    """
    # 1. TF-IDF 벡터화
    texts = document_df[text_column]
    vectorizer = TfidfVectorizer(max_features=max_features, stop_words='english')
    X = vectorizer.fit_transform(texts)

    # 2. 차원 축소 (SVD)
    svd = TruncatedSVD(n_components=svd_dim, random_state=42)
    X_reduced = svd.fit_transform(X)

    # 3. 여러 k 값 평가
    scores = {}
    for k in range(k_range[0], k_range[1] + 1):
        kmeans = KMeans(n_clusters=k, random_state=42)
        labels = kmeans.fit_predict(X_reduced)
        
        if len(set(labels)) > 1:
            sil = silhouette_score(X_reduced, labels)
            db = davies_bouldin_score(X_reduced, labels)
            ch = calinski_harabasz_score(X_reduced, labels)
            scores[k] = {"silhouette": sil, "davies_bouldin": db, "calinski": ch}
        else:
            scores[k] = {"silhouette": None, "davies_bouldin": None, "calinski": None}

    # 4. 지표 정규화 및 최적 k 선택
    sil_values = [v["silhouette"] for v in scores.values() if v["silhouette"] is not None]
    db_values  = [v["davies_bouldin"] for v in scores.values() if v["davies_bouldin"] is not None]
    ch_values  = [v["calinski"] for v in scores.values() if v["calinski"] is not None]

    def normalize(values, higher_is_better=True):
        arr = np.array(values)
        if higher_is_better:
            return (arr - arr.min()) / (arr.max() - arr.min())
        else:
            return (arr.max() - arr) / (arr.max() - arr.min())

    sil_norm = normalize(sil_values, higher_is_better=True)
    db_norm  = normalize(db_values, higher_is_better=False)
    ch_norm  = normalize(ch_values, higher_is_better=True)

    combined_scores = sil_norm + db_norm + ch_norm
    valid_keys = [k for k,v in scores.items() if v["silhouette"] is not None]
    best_index = np.argmax(combined_scores)
    best_k = valid_keys[best_index]

    # 5. 최적 k로 KMeans 실행
    final_kmeans = KMeans(n_clusters=best_k, random_state=42)
    labels = final_kmeans.fit_predict(X_reduced)
    document_df['Cluster'] = labels

    # 6. 클러스터별 대표 문서 및 유사 문서 Top-N 추출
    similarity_matrix = cosine_similarity(X)
    representatives = {}

    for cluster_id in range(best_k):
        cluster_docs = document_df[document_df['Cluster'] == cluster_id]
        if cluster_docs.empty:
            continue
        
        # 클러스터 내 첫 번째 문서를 대표 문서로 선택 (다른 기준도 가능)
        rep_idx = cluster_docs.index[0]
        rep_text = cluster_docs[text_column].iloc[0]
        
        # 대표 문서와 유사 문서 Top-N
        sim_scores = similarity_matrix[rep_idx]
        top_indices = np.argsort(sim_scores)[-top_n-1:-1]  # 자기 자신 제외
        similar_docs = document_df.iloc[top_indices][[text_column, 'Cluster']]
        
        representatives[cluster_id] = {
            "representative": rep_text,
            "similar_docs": similar_docs
        }

    return {
        "scores": scores,
        "best_k": best_k,
        "best_metrics": scores[best_k],
        "combined_score": combined_scores[best_index],
        "representatives": representatives
    }

In [4]:
result = auto_kmeans_with_representatives(document_df, text_column='processed_text', top_n=5)

print("=== 최적 k 선택 결과 ===")
print(f"Best k = {result['best_k']}")
print(f"Silhouette = {result['best_metrics']['silhouette']:.3f}")
print(f"Davies-Bouldin = {result['best_metrics']['davies_bouldin']:.3f}")
print(f"Calinski-Harabasz = {result['best_metrics']['calinski']:.3f}")
print(f"Combined Score = {result['combined_score']:.3f}")

print("\n=== 클러스터별 대표 문서 및 유사 문서 ===")
for cluster_id, info in result['representatives'].items():
    print(f"\nCluster {cluster_id}:")
    print("대표 문서:", info['representative'])
    print("유사 문서 Top-N:")
    print(info['similar_docs'])

=== 최적 k 선택 결과 ===
Best k = 10
Silhouette = 0.152
Davies-Bouldin = 1.789
Calinski-Harabasz = 2.764
Combined Score = 2.000

=== 클러스터별 대표 문서 및 유사 문서 ===

Cluster 0:
대표 문서: think new keyboard rival great hp mini keyboard since battery life difference minimum reason upgrade would get better keyboard keyboard good hp samsung netbooks keyboard size found standard computer machine run little hot keyboard large enough accommodate touch typing ease home asus connected desktop keyboard monitor however since lot word processing keyboard touchpad button real concern keyboard responsive feel larger key along oversize right shift key real plus light weight battery life keyboard non protruding removable battery bright screen track pad disable button general ease use make hard beat battery life continues hold keyboard true godsend full size key tiny light slightly smaller standard keyboard problem use touch typist key good size feel good keyboard quite nice large hand make lot fewer mistake fit finish